In [1]:
import HTSeq
import numpy as np
from collections import Counter
from fractions import Fraction

In [2]:
def generate_sam_file(
    regions: dict, 
    relative_translation: dict,
    downstream_cleavage_model: list,
    upstream_cleavage_model: list,
    untemplated_addition_probability: float=.1,
    read_number: int=1_000,
    sam_file_name: str='ORF_test.sam',
    seed: int=42,
    fasta_path: str='ORF_test.fasta'
    ):

    upstream_cleavage_model = upstream_cleavage_model / sum(upstream_cleavage_model)
    downstream_cleavage_model = downstream_cleavage_model / sum(downstream_cleavage_model)
    
    with open(fasta_path, 'r') as f:
        f.readline()
        sequence = f.readline().strip()
    np.random.seed(seed)

    region_names = list(regions.keys())
    read_probabilities_per_region = [regions[k].iv.length * relative_translation[k] for k in region_names]
    read_probabilities_per_region = np.array(read_probabilities_per_region) / sum(read_probabilities_per_region)
    
    read_sources = np.random.choice(region_names, p=read_probabilities_per_region, size=read_number)
    read_untemplated_additions = np.random.binomial(1, untemplated_addition_probability, size=read_number)
    read_upstream_cleavages = np.random.choice(len(upstream_cleavage_model), p=upstream_cleavage_model, size=read_number)
    read_downstream_cleavages = np.random.choice(len(downstream_cleavage_model), p=downstream_cleavage_model, size=read_number)

    read_indices = [f'r{i:0{len(str(read_number))}}' for i in range(read_number)]
    bitwise_flag = 0
    rname = 'chr1'
    mapq = 0
    rnext = '*'
    pnext = 0
    tlen = 0
    qual = '*'
    nh = 'NH:i:1'

    with open(sam_file_name, 'w') as f:
        f.write('@HD	VN:1.6	SO:coordinate\n@SQ	SN:chr1	LN:3000\n')
        for i in range(read_number):
            read_source = f'CO:Z:{read_sources[i]}'
            if regions[read_sources[i]].type == 'ORF':
                p_site = np.random.choice(np.arange(regions[read_sources[i]].iv.start, regions[read_sources[i]].iv.end-2,3))
            else:
                p_site = np.random.choice(np.arange(regions[read_sources[i]].iv.start, regions[read_sources[i]].iv.end-2))
            start = p_site - read_upstream_cleavages[i] - read_untemplated_additions[i]
            length = 3 + read_upstream_cleavages[i] + read_downstream_cleavages[i] + read_untemplated_additions[i]
            seq = sequence[p_site - read_upstream_cleavages[i]:p_site + read_downstream_cleavages[i] + 3]

            if read_untemplated_additions[i] == 1: # case has untemplated addition
                untemplated_nucleotide = np.random.choice(['A', 'C', 'G', 'T'])

                seq = untemplated_nucleotide + seq
                if untemplated_nucleotide != sequence[p_site - read_upstream_cleavages[i] - 1]:
                    # case untemplated addition is observed
                    mdz = f'MD:Z:0{untemplated_nucleotide}{length-1}'
                    start += 1
                else:
                    # case untemplated addition is not observed
                    mdz = f'MD:Z:{length}'
            else:
                mdz = f'MD:Z:{length}'

            cigar = f'{length}M'

            sam_line = f'{read_indices[i]}\t{bitwise_flag}\t{rname}\t{start+1}\t{mapq}\t{cigar}\t{rnext}\t{pnext}\t{tlen}\t{seq}\t{qual}\t{nh}\t{mdz}\t{read_source}\n'

            f.write(sam_line)

    c = Counter(read_sources)
    s = sum([c[k]/regions[k].iv.length for k in regions])
    for k in regions:
        print(f'{k}\t{c[k]/regions[k].iv.length/s:.4f}')


In [3]:
#gtf_path = 'ORF_NOISE_test.gtf'
gtf_path = 'reg_test.gtf'
#gtf_path = 'ad1 copy.gtf'


regions = {}
for region in HTSeq.GFF_Reader(gtf_path):
    regions[region.attr['region_id']] = region


region_relative_translation = {region.attr['region_id']: float(Fraction(region.attr['activity'])) for region in HTSeq.GFF_Reader(gtf_path)}


generate_sam_file(
    regions=regions,
    relative_translation=region_relative_translation,
    downstream_cleavage_model=np.loadtxt('downstream_cleavage_model', dtype=float),
    upstream_cleavage_model=np.loadtxt('upstream_cleavage_model', dtype=float),
    untemplated_addition_probability=0.1080613528993745,
    read_number=100_000,
    sam_file_name='test.sam',
    seed=10,
    fasta_path='ORF_test.fasta'
)

orf1	0.0236
orf2	0.4464
orf3	0.2213
orf4	0.1139
orf5	0.0000
orf6	0.1065
orf7	0.0444
orf8	0.0217
orf9	0.0000
orf10	0.0000
orf11	0.0000
orf12	0.0000
orf13	0.0000
orf14	0.0000
orf15	0.0000
orf16	0.0000
orf17	0.0000
orf18	0.0000
noise1	0.0223
